In [1]:
# standardní knihovny Pythonu
import sys
import time
import re
from pathlib import Path
from urllib.parse import urljoin

# knihovny třetích stran
import pandas as pd
import requests
from bs4 import BeautifulSoup

# nastavení root složky projektu pro import utility modulů
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sdílené konstanty a společné schéma datasetu
from utils.constants import HEADERS
from utils.schema import FINAL_COLUMNS

# základní technické čištění textu
from utils.text_cleaning import (
    clean_base_text,
    join_text_parts
)

In [2]:
# akademický rok použitý ve zdrojových URL
ACADEMIC_YEAR = "2025/26"

# základní doména katalogu DRPS
BASE_DOMAIN = "https://www.drps.ed.ac.uk"

# stránky jednotlivých studijních oblastí
SUBJECT_PAGES = {
    "Business Studies": "https://www.drps.ed.ac.uk/25-26/dpt/cx_sb_bust.htm",
    "Economics": "https://www.drps.ed.ac.uk/25-26/dpt/cx_sb_ecnm.htm",
}

In [3]:
# extrakce SCQF levelu z nadpisu sekce
def parse_scqf_level(text):

    if pd.isna(text):
        return pd.NA

    match = re.search(
        r"SCQF Level\s+0?(\d+)",
        str(text)
    )

    if not match:
        return pd.NA

    return int(match.group(1))


# identifikace bakalářských úrovní studia
def is_undergraduate_level(level):

    # SCQF level 11 odpovídá postgraduate úrovni
    return level in [8, 9, 10]


# normalizace semestru do společných kategorií
def normalize_semester(period):

    if pd.isna(period):
        return pd.NA

    period_lower = str(period).lower()

    if "semester 1" in period_lower:
        return "winter"

    if "semester 2" in period_lower:
        return "summer"

    if "full year" in period_lower:
        return "full_year"

    return pd.NA

In [4]:
# parsování jednoho řádku tabulky předmětů
def parse_course_row(
    row,
    subject_area,
    source_list_url,
    level
):

    cols = row.find_all("td")

    if len(cols) < 5:
        return None

    course_code = clean_base_text(
        cols[0].get_text(" ", strip=True)
    )

    availability = clean_base_text(
        cols[1].get_text(" ", strip=True)
    )

    course_link = cols[2].find(
        "a",
        href=True
    )

    period = clean_base_text(
        cols[3].get_text(" ", strip=True)
    )

    credits_raw = clean_base_text(
        cols[4].get_text(" ", strip=True)
    )

    if pd.isna(course_code) or not course_link:
        return None

    course_title = clean_base_text(
        course_link.get_text(" ", strip=True)
    )

    detail_url = urljoin(
        source_list_url,
        course_link["href"]
    )

    credits_match = re.search(
        r"\d+",
        str(credits_raw)
    )

    credits = (
        int(credits_match.group(0))
        if credits_match
        else pd.NA
    )

    # SV a VV označují předměty dostupné výměnným studentům
    if (
        pd.isna(availability)
        or not availability.startswith(("SV", "VV"))
    ):
        return None

    # zahrnuty jsou pouze bakalářské úrovně studia
    if not is_undergraduate_level(level):
        return None

    return {
        "subject_area": subject_area,
        "course_code": course_code,
        "course_title": course_title,
        "availability": availability,
        "period": period,
        "semester_normalized": normalize_semester(period),
        "credits": credits,
        "level": level,
        "detail_url": detail_url,
        "source_list_url": source_list_url,
        "academic_year": ACADEMIC_YEAR,
    }

In [5]:
# získání odkazů na detailní stránky předmětů z přehledové stránky DRPS
def collect_edinburgh_course_links(subject_area, list_url):

    try:
        response = requests.get(
            list_url,
            headers=HEADERS,
            timeout=20
        )

        response.raise_for_status()
        response.encoding = response.apparent_encoding

    except requests.RequestException as e:
        print(f"CHYBA při načítání seznamu {subject_area}: {e}")
        return []

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    courses = []
    current_level = pd.NA

    for element in soup.find_all(["h3", "tr"]):

        if (
            element.name == "h3"
            and "scqf_level" in element.get("class", [])
        ):
            current_level = parse_scqf_level(
                element.get_text(" ", strip=True)
            )
            continue

        if element.name == "tr":

            parsed = parse_course_row(
                row=element,
                subject_area=subject_area,
                source_list_url=list_url,
                level=current_level
            )

            if parsed is not None:
                courses.append(parsed)

    return courses

In [6]:
# sběr odkazů na vhodné předměty z jednotlivých oblastí
all_courses = []

for subject_area, list_url in SUBJECT_PAGES.items():

    print(f"Stahuji seznam: {subject_area}")

    subject_courses = collect_edinburgh_course_links(
        subject_area,
        list_url
    )

    print(f"Nalezeno vhodných kurzů: {len(subject_courses)}")

    all_courses.extend(subject_courses)

    # omezení frekvence requestů vůči serveru
    time.sleep(1)

edinburgh_links_df = pd.DataFrame(all_courses)

print(f"Celkem nalezených kurzů: {len(edinburgh_links_df)}")

Stahuji seznam: Business Studies
Nalezeno vhodných kurzů: 61
Stahuji seznam: Economics
Nalezeno vhodných kurzů: 38
Celkem nalezených kurzů: 99


In [7]:
# názvy sekcí používané jako hranice při extrakci textu
SECTION_LABELS = [
    "Summary",
    "Course description",
    "Entry Requirements (not applicable to Visiting Students)",
    "Information for Visiting Students",
    "Course Delivery Information",
    "Assessment",
    "Additional Information (Assessment)",
    "Feedback",
    "Exam Information",
    "Learning Outcomes",
    "Reading List",
    "Graduate Attributes and Skills",
    "Keywords",
    "Contacts",
]


# převod stránky na očištěné textové řádky
def get_page_text_lines(soup):

    text = soup.get_text("\n", strip=True)

    lines = [
        clean_base_text(line)
        for line in text.split("\n")
    ]

    return [
        line
        for line in lines
        if not pd.isna(line)
    ]


# extrakce textu mezi dvěma sekčními nadpisy
def extract_section(
    lines,
    start_label,
    stop_labels=SECTION_LABELS
):

    if start_label not in lines:
        return pd.NA

    start_idx = lines.index(start_label) + 1
    end_idx = len(lines)

    for i in range(start_idx, len(lines)):

        if lines[i] in stop_labels:
            end_idx = i
            break

    section_lines = lines[start_idx:end_idx]

    return clean_base_text(
        " ".join(section_lines)
    )


# spojení více sekcí do jednoho pole
def combine_sections(*sections):

    return join_text_parts(sections)


# odstranění technického nadpisu z literatury
def clean_literature(text):

    if pd.isna(text):
        return pd.NA

    text = str(text).replace(
        "Additional Information",
        ""
    )

    return clean_base_text(text)


# odstranění technické poznámky z hodnocení
def clean_assessment(text):

    if pd.isna(text):
        return pd.NA

    text = str(text).replace(
        "(Further Info)",
        ""
    )

    return clean_base_text(text)

In [8]:
# parsování detailní stránky předmětu
def parse_edinburgh_detail_page(row):

    detail_url = row["detail_url"]

    response = requests.get(
        detail_url,
        headers=HEADERS,
        timeout=20
    )

    response.raise_for_status()
    response.encoding = response.apparent_encoding

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    lines = get_page_text_lines(soup)

    summary = extract_section(
        lines,
        "Summary"
    )

    course_description = extract_section(
        lines,
        "Course description"
    )

    learning_outcomes = extract_section(
        lines,
        "Learning Outcomes"
    )

    reading_list = clean_literature(
        extract_section(lines, "Reading List")
    )

    assessment = clean_assessment(
        extract_section(lines, "Assessment")
    )

    credits = row["credits"]

    ects_credits = (
        credits / 2
        if not pd.isna(credits)
        else pd.NA
    )

    return {
        "course_title": row["course_title"],
        "course_code": row["course_code"],
        "university": "University of Edinburgh",
        "country": "United Kingdom",
        "source_url": detail_url,
        "academic_year": row["academic_year"],
        "semester": row["period"],
        "semester_normalized": row["semester_normalized"],
        "study_level": "Undergraduate",
        "credits": credits,
        "credit_system": "SCQF credits",
        "ects_credits": ects_credits,
        "aims": summary,
        "course_content": course_description,
        "learning_outcomes": learning_outcomes,
        "assessment_methods": assessment,
        "literature": reading_list,
    }

In [9]:
# stažení a naparsování detailních stránek předmětů
edinburgh_rows = []
edinburgh_errors = []

for idx, row in edinburgh_links_df.iterrows():

    print(f"{idx + 1}/{len(edinburgh_links_df)} | {row['course_code']}")

    try:
        record = parse_edinburgh_detail_page(row)
        edinburgh_rows.append(record)

    except Exception as e:
        edinburgh_errors.append({
            "course_code": row["course_code"],
            "detail_url": row["detail_url"],
            "error": str(e),
        })

    # omezení frekvence requestů vůči serveru
    time.sleep(1)

df_edinburgh = pd.DataFrame(edinburgh_rows)
edinburgh_errors_df = pd.DataFrame(edinburgh_errors)

print("Stažené předměty:", len(df_edinburgh))
print("Chyby:", len(edinburgh_errors_df))

1/99 | BUST08018
2/99 | BUST08032
3/99 | BUST08005
4/99 | BUST08056
5/99 | BUST08033
6/99 | BUST08031
7/99 | BUST08027
8/99 | BUST08053
9/99 | BUST08009
10/99 | BUST08008
11/99 | BUST08004
12/99 | BUST08054
13/99 | BUST08028
14/99 | BUST08003
15/99 | BUST08051
16/99 | BUST08057
17/99 | BUST08045
18/99 | BUST08042
19/99 | BUST10129
20/99 | ACCN10026
21/99 | BUST10127
22/99 | BUST10004
23/99 | BUST10007
24/99 | BUST10012
25/99 | BUST10162
26/99 | BUST10133
27/99 | BUST10144
28/99 | BUST10165
29/99 | BUST10153
30/99 | BUST10124
31/99 | BUST10120
32/99 | BUST10163
33/99 | BUST10155
34/99 | BUST10023
35/99 | BUST10148
36/99 | BUST10157
37/99 | BUST10150
38/99 | BUST10018
39/99 | BUST10067
40/99 | BUST10104
41/99 | BUST10032
42/99 | BUST10135
43/99 | BUST10113
44/99 | BUST10107
45/99 | BUST10028
46/99 | BUST10166
47/99 | BUST10108
48/99 | BUST10134
49/99 | BUST10149
50/99 | BUST10092
51/99 | BUST10172
52/99 | BUST10145
53/99 | BUST10024
54/99 | BUST10142
55/99 | CMSE10002
56/99 | BUST10168
5

In [10]:
# sjednocení pořadí sloupců podle společného schématu
df_edinburgh = df_edinburgh[FINAL_COLUMNS]

print(f"Počet předmětů: {len(df_edinburgh)}")

Počet předmětů: 99


In [12]:
# export naparsovaných Edinburgh sylabů do interim vrstvy
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "edinburgh_syllabi_raw_parsed.csv"
)

df_edinburgh.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Uloženo {len(df_edinburgh)} předmětů do {OUTPUT_PATH.name}")

Uloženo 99 předmětů do edinburgh_syllabi_raw_parsed.csv
